### 速览
Step1 依赖 - Step2 数据 - Step3 训练 - Step4 混淆图 - Step5 指标 - Step6 识别器重放


## Step 1
检查本机/虚拟环境与工程模块能否 `import`。


In [ ]:
import sys, platform
print('Python     :', sys.version.split()[0], '  OS:', platform.system(), platform.release())

ok = True
for mod in ('numpy', 'scipy', 'sklearn', 'joblib', 'PyQt5.QtCore'):
    try:
        __import__(mod)
        print(f'  [ok] {mod}')
    except Exception as e:
        print(f'  [fail] {mod}  -- {e}')
        ok = False

for mod in ('adaptive_preprocessing', 'ml_activity_features',
            'ml_branch_models', 'gait_knee_features', 'ml_train_branch_rfs',
            'personal_calibration', 'realtime_recognizer',
            'foot_pressure_monitor'):
    try:
        __import__(mod)
        print(f'  [ok] {mod} (project)')
    except Exception as e:
        print(f'  [fail] {mod} (project)  -- {e}')
        ok = False

print('\nOverall: ', 'OK — proceed' if ok else 'missing packages — install dependencies')

## Step 2
`labeled_csv_paths` 与 `load_csv_files`：行数、通道、标签占比。


In [ ]:
import os
import ml_activity_features as maf

_project_root = os.path.dirname(os.path.abspath(maf.__file__))
if os.path.normcase(os.path.abspath(os.getcwd())) != os.path.normcase(
    os.path.abspath(_project_root)
):
    os.chdir(_project_root)

print('DATA_DIR =', maf.DATA_DIR)
files = maf.labeled_csv_paths()
print(f'Found under saving_data/:  {len(files)}  *sensor_data_dual_labeled_*.csv file(s)\n')
for p in files[:6]:
    print('  ', os.path.basename(p))
if len(files) > 6:
    print(f'  ... (omitted {len(files) - 6} files)')
    
raw, labels, subjects, mode = maf.load_csv_files(maf.DATA_DIR, labeled_only=True, raw_adc=True)
if not files or raw.size == 0 or len(labels) == 0:
    raise RuntimeError("无标注行：请在 saving_data/ 中放入 *sensor_data_dual_labeled_*.csv。")
print(f'\nTotal rows: {raw.shape[0]}  ≈ {raw.shape[0]/maf.SAMPLE_HZ:.0f} s  '
      f'({raw.shape[0]/maf.SAMPLE_HZ/60:.1f} min of data)')
print(f'Channels: {raw.shape[1]}  (order: L_Forefoot, L_Heel, L_Knee, R_Forefoot, R_Heel, R_Knee (8-col CSV drops toe here)')

from collections import Counter
print('\nLabel counts: ')
for lab, n in Counter(labels).most_common():
    print(f'  {lab:25s}  {n:>5d}  ({n/len(labels)*100:5.1f}%)')

print(f'\nNormalization stats are computed on all  {len(files)} CSV(s) concatenated; ')
print('   not per-file or first-N rows.')

## Step 3
运行 `ml_train_branch_rfs` 主流程，检查 `rf_*.joblib` 与 `personal_calibration.json`。


In [ ]:
import importlib, ml_train_branch_rfs as mtb, ml_branch_models as mbm
importlib.reload(mtb)

exit_code = mtb.main(argv=[])
print('\nTrain script exit code: ', exit_code, '(0 = success)')
print('\nExpect four model files:')
for fn in mbm.BRANCH_TO_FILE.values():
    size = os.path.getsize(fn) / 1024 if os.path.isfile(fn) else 0
    print(f'  {fn:34s}  {"OK" if size > 0 else "missing"}  ({size:.1f} KB)')

print('\nAnd calibration JSON:')
print('  personal_calibration.json  OK' if os.path.isfile('personal_calibration.json') else '  MISSING')

## Step 4
展示各分支 `confusion_*_*.png`（若已训练生成）。


In [ ]:
from IPython.display import Image, display, Markdown

BRANCHES = ['stairs', 'sitting', 'walking', 'standing']
cm_dir = 'confusion_matrices'

for br in BRANCHES:
    display(Markdown(f'#### · Branch = `{br}`'))
    candidates = [
        os.path.join(cm_dir, f'confusion_{br}_blocked_oof.png'),
        os.path.join(cm_dir, f'confusion_{br}_lofo_oof.png'),
        os.path.join(cm_dir, f'confusion_{br}_val.png'),
        os.path.join(cm_dir, f'confusion_{br}_test.png'),
    ]
    shown = False
    for p in candidates:
        if os.path.isfile(p):
            display(Image(filename=p, width=560))
            shown = True
            break
    if not shown:
        print(f'(no PNG for {br}  confusion PNG — run training first)')
    print()

## Step 5
从 `rf_*.joblib` 打印 blocked/LOFO 等评估列说明。


In [ ]:
import joblib
import numpy as np

print(f'{"branch":<18}{"N":>7}{"segs":>6}{"folds":>6}'
      f'{"mean":>9}{"std":>9}{"min":>9}{"OOF acc":>10}{"eval":>22}')
print('-' * 96)
for br, fn in mbm.BRANCH_TO_FILE.items():
    if not os.path.isfile(fn):
        print(f'{br:<18}  (no model file)')
        continue
    m = joblib.load(fn).get('metrics', {})
    if not m:
        print(f'{br:<18}  (metrics missing — retrain)')
        continue
    ev = m.get('evaluation', 'unknown')
    if ev == 'blocked_intra_file':
        print(f'{br:<18}{m["n_samples"]:>7}{m["blocked_n_segments"]:>6}'
              f'{m["blocked_n_folds"]:>6}'
              f'{m["blocked_mean"]:>9.4f}{m["blocked_std"]:>9.4f}'
              f'{m["blocked_min"]:>9.4f}{m["oof_accuracy"]:>10.4f}'
              f'{ev:>22}')
    elif ev == 'leave_one_file_out':
        print(f'{br:<18}{m["n_samples"]:>7}{"-":>6}{m["lofo_n_files"]:>6}'
              f'{m["lofo_mean"]:>9.4f}{m["lofo_std"]:>9.4f}'
              f'{m["lofo_min"]:>9.4f}{m["oof_accuracy"]:>10.4f}'
              f'{ev:>22}')
    elif 'val_accuracy' in m:
        print(f'{br:<18}{m["n_samples"]:>7}  (random split)  val={m["val_accuracy"]:.4f}'
              f'  test={m["test_accuracy"]:.4f}')
    else:
        print(f'{br:<18}  unknown eval mode: {ev}')
print()
print('Columns (blocked CV):')
print('  N        = windows in branch')
print('  segs     = label-segments (same file, same label)')
print('  folds    = CV folds (default 5)')
print('  mean     = mean fold accuracy (weighted)')
print('  std      = std across folds')
print('  min      = worst-fold accuracy')
print('  OOF acc  = pooled OOF accuracy')

## Step 6
`OnlineRecognizer` 全序列重放，与 CSV 标签比准确率（排除 UNKNOWN 等）。


In [ ]:
import realtime_recognizer as rrz
from collections import Counter

rec = rrz.OnlineRecognizer(calibration='personal_calibration.json')
print(f'Recognizer ready: bank frozen={all(c._freeze for c in rec._adapt_bank.channels)}')
print(f'            global calibration loaded: {rec._calibration.has_global_stats}')

pred, gt = [], []
for i in range(raw.shape[0]):
    lf, lh, lk, rf, rh, rk = map(float, raw[i])
    out = rec.update_bilateral((lf, lh, lk), (rf, rh, rk), t=i / maf.SAMPLE_HZ)
    pred.append(out['state'])
    gt.append(labels[i])

valid = [(p, g) for p, g in zip(pred, gt) if g not in ('UNKNOWN', 'SIT_TO_STAND')]
match = sum(1 for p, g in valid if p == g)
print(f'\nOffline replay accuracy =  {match / max(1, len(valid)):.1%}  '
      f'（{match} / {len(valid)}  rows, excl. UNKNOWN / SIT_TO_STAND)')
print('\nPredicted state dist (top 10):')
for s, n in Counter(pred).most_common(10):
    print(f'  {s:25s}  {n}')